<a href="https://colab.research.google.com/github/eng20260311/AIFFEL_quest_eng/blob/master/NLP/NLP03/chatbot_transformer_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 한국어 챗봇 — Transformer + RAG 평가

이 노트북은 다음을 포함합니다.

- Step 1: 데이터 다운로드
- Step 2: 데이터 정제 (`preprocess_sentence`)
- Step 3: 토큰화 (`build_corpus` + Mecab)
- Step 4: Lexical Substitution Augmentation (Word2Vec ko.bin)
- Step 5: 벡터화 (`<start>`, `<end>` 토큰, 공유 단어사전)
- Step 6: Transformer 학습 (Untitled24에서 가져온 구현 재사용)
- Step 7: BLEU 점수 계산 + **RAG 스타일 retrieval 챗봇과 비교**

Colab에서 위에서부터 순서대로 실행하세요. GPU 런타임을 권장합니다.


## Step 0. 환경 설정

Colab에 mecab(은전한닢)과 필요한 라이브러리를 설치합니다.

In [1]:
# Mecab 설치 (Colab 한정, 약 1~2분 소요)
!pip install -q konlpy
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 16.8 MB/s eta 0:00:00
Install mecab-ko
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1381k  100 1381k    0     0   437k      0  0:00:03  0:00:03 --:--:--  727k
mecab-0.996-ko-0.9.2/
mecab-0.996-ko-0.9.2/example/
mecab-0.996-ko-0.9.2/example/example.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.c
mecab-0.996-ko-0.9.2/example/example.c
mecab-0.996-ko-0.9.2/example/thread_test.cpp
mecab-0.996-ko-0.9.2/mecab-config.in
mecab-0.996-ko-0.9.2/man/
mecab-0.996-ko-0.9.2/man/Makefile.am
mecab-0.996-ko-0.9.2/man/mecab.1
mecab-0.996-ko-0.9.2/man/Makefile.in
mecab-0.996-ko-0.9.2/mecab.iss.in
mecab-0.996-ko-0.9.2/config.gu

In [9]:
# 나머지 라이브러리
!pip install -q gensim sentencepiece nltk tqdm
!pip install -q python-mecab-ko


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 35.2 MB/s eta 0:00:00


In [10]:
# 공통 import
import os, re, random, math, time, urllib.request, io, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
torch.manual_seed(42); random.seed(42); np.random.seed(42)


device: cuda


## Step 1. 데이터 다운로드

`songys/Chatbot_data`의 `ChatbotData.csv`를 사용합니다.

In [11]:
import pandas as pd

URL = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
df = pd.read_csv(URL)
print("총 페어 수:", len(df))
df.head()


총 페어 수: 11823


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [12]:
questions = df["Q"].tolist()       #질문 변수 생성
answers   = df["A"].tolist()       #답변 변수 생성
print(len(questions), len(answers))
print(questions[:3])
print(answers[:3])


11823 11823
['12시 땡!', '1지망 학교 떨어졌어', '3박4일 놀러가고 싶다']
['하루가 또 가네요.', '위로해 드립니다.', '여행은 언제나 좋죠.']


## Step 2. `preprocess_sentence`

- 영문자는 소문자로 변환
- 한글, 영문, 숫자, 주요 특수문자(`.?!,`)만 남기고 제거
- 토크나이저(mecab)가 구두점 분리를 알아서 해주므로 따로 공백 패딩은 하지 않음


In [14]:
def preprocess_sentence(sentence):
    # 1) 영문 소문자화
    sentence = sentence.lower()
    # 2) 허용 문자만 남기고 나머지는 공백으로
    #    - 한글(가-힣), 영문(a-z), 숫자(0-9), 주요 특수문자(.?!,)
    sentence = re.sub(r"[^a-z0-9가-힣?.!,]+", " ", sentence)
    # 3) 연속 공백 압축 + 양끝 공백 제거
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

# 간단 테스트
print(preprocess_sentence("안녕하세요!! 오늘 너~~무 신난다 ^_^ 123"))
print(preprocess_sentence("Hello, World!! 반가워요 :)"))


안녕하세요!! 오늘 너 무 신난다 123
hello, world!! 반가워요


## Step 3. `build_corpus` (Mecab 토큰화)

- 입력: 소스/타깃 리스트, 토크나이즈 함수(`mecab.morphs`)
- `preprocess_sentence`로 정제 후 토큰화
- 일정 길이 이상 토큰은 제외
- **중복은 소스/타깃 각각 따로 검사** — 쌍이 깨지면 안 되므로 인덱스 단위로 처리


In [15]:
from mecab import MeCab
mecab = MeCab()
print(mecab.morphs("간만에 여자친구랑 데이트 하기로 했어"))

['간만에', '여자', '친구', '랑', '데이트', '하', '기', '로', '했', '어']


In [16]:
def build_corpus(src_list, tgt_list, tokenize_fn, max_len=30):
    """
    - src_list, tgt_list: 같은 길이의 병렬 리스트
    - tokenize_fn: 토큰화 함수 (예: mecab.morphs)
    - max_len: 토큰 개수가 이 값을 넘으면 페어 제외
    반환: (src_corpus, tgt_corpus) — 둘 다 토큰 리스트의 리스트, 길이 동일
    """
    assert len(src_list) == len(tgt_list)

    seen_src = set()
    seen_tgt = set()
    src_corpus, tgt_corpus = [], []

    for s, t in zip(src_list, tgt_list):
        s_clean = preprocess_sentence(s)
        t_clean = preprocess_sentence(t)

        # 소스/타깃 각각의 원문 기준 중복 제거 (쌍을 흐트러뜨리지 않음)
        if s_clean in seen_src or t_clean in seen_tgt:
            continue

        s_tok = tokenize_fn(s_clean)
        t_tok = tokenize_fn(t_clean)

        if len(s_tok) == 0 or len(t_tok) == 0:
            continue
        if len(s_tok) > max_len or len(t_tok) > max_len:
            continue

        seen_src.add(s_clean)
        seen_tgt.add(t_clean)
        src_corpus.append(s_tok)
        tgt_corpus.append(t_tok)

    return src_corpus, tgt_corpus


que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs, max_len=30)
print("페어 수:", len(que_corpus))
print("샘플:", que_corpus[0], "->", ans_corpus[0])


페어 수: 7730
샘플: ['12', '시', '땡', '!'] -> ['하루', '가', '또', '가', '네요', '.']


## Step 4. Augmentation (Lexical Substitution)

`Kyubyong/wordvectors`의 한국어 Word2Vec (`ko.bin`)을 사용해 일부 단어를 가장 유사한 단어로 치환합니다.

- 원본의 약 3배가 되도록 다음 둘을 합칩니다.
  1. (Aug(que), ans)
  2. (que, Aug(ans))
- 원본도 그대로 유지 → 총 3배

In [19]:
# Kyubyong/wordvectors에서 한국어 Word2Vec(ko.zip) 다운로드
# 주의: 원 저장소가 Dropbox 링크를 사용해 시점에 따라 받기 어려울 수 있습니다.
#       실패하면 아래 셀(대안)로 넘어가세요.
import os
os.makedirs("/content/wv", exist_ok=True)

try:
    url = "https://github.com/Kyubyong/wordvectors/raw/master/pretrained/ko.zip"
    urllib.request.urlretrieve(url, "/content/wv/ko.zip")
    with zipfile.ZipFile("/content/wv/ko.zip") as z:
        z.extractall("/content/wv")
    print(os.listdir("/content/wv"))
except Exception as e:
    print("다운로드 실패:", e)
    print("→ 다음 셀의 대안(FastText/임시 임베딩)을 사용하세요.")


다운로드 실패: HTTP Error 404: Not Found
→ 다음 셀의 대안(FastText/임시 임베딩)을 사용하세요.


In [20]:
# Word2Vec 로드 (성공한 경우)
from gensim.models import Word2Vec, KeyedVectors

W2V = None
for path in ["/content/wv/ko.bin", "/content/wv/ko/ko.bin"]:
    if os.path.exists(path):
        try:
            W2V = Word2Vec.load(path).wv
            print("Word2Vec 로드 완료:", path, "vocab:", len(W2V.key_to_index))
            break
        except Exception:
            try:
                W2V = KeyedVectors.load(path)
                print("KeyedVectors 로드 완료:", path)
                break
            except Exception as e:
                print("로드 실패:", path, e)

if W2V is None:
    print("⚠️ Word2Vec 로드 실패 — Augmentation 없이 진행합니다.")


⚠️ Word2Vec 로드 실패 — Augmentation 없이 진행합니다.


In [21]:
def lexical_sub(tokens, model, p=0.3, topn=5):
    """
    토큰 리스트에서 일부 토큰을 모델의 유사어로 치환.
    - p: 토큰별 치환 확률
    - topn: 후보 중 무작위 1개 선택
    """
    if model is None:
        return list(tokens)
    new = []
    for tok in tokens:
        if random.random() < p and tok in model.key_to_index:
            try:
                candidates = [w for w, _ in model.most_similar(tok, topn=topn)]
                if candidates:
                    new.append(random.choice(candidates))
                    continue
            except KeyError:
                pass
        new.append(tok)
    return new


def augment_corpus(que_corpus, ans_corpus, model):
    aug_que = [lexical_sub(q, model) for q in que_corpus]
    aug_ans = [lexical_sub(a, model) for a in ans_corpus]

    # (1) (Aug(que), ans) + (2) (que, Aug(ans))   ← 원본은 별도 합산
    src = list(que_corpus) + aug_que + list(que_corpus)
    tgt = list(ans_corpus) + list(ans_corpus) + aug_ans
    return src, tgt


src_corpus, tgt_corpus = augment_corpus(que_corpus, ans_corpus, W2V)
print(f"원본 {len(que_corpus)} → 증강 후 {len(src_corpus)}  (약 {len(src_corpus)/len(que_corpus):.1f}배)")


원본 7730 → 증강 후 23190  (약 3.0배)


## Step 5. 벡터화

- 타깃에 `<start>` / `<end>` 토큰 추가
- 소스/타깃이 같은 언어이므로 **공유 단어사전** 구축
- 패딩하여 `enc_train`, `dec_train` 생성

In [22]:
# 1) 타깃에 특수 토큰 추가
tgt_corpus = [["<start>"] + a + ["<end>"] for a in tgt_corpus]
print(tgt_corpus[0])


['<start>', '하루', '가', '또', '가', '네요', '.', '<end>']


In [23]:
# 2) 공유 단어사전
PAD, UNK, START, END = "<pad>", "<unk>", "<start>", "<end>"
specials = [PAD, UNK, START, END]

from collections import Counter
counter = Counter()
for s in src_corpus: counter.update(s)
for t in tgt_corpus: counter.update(t)

# 빈도 상위만 유지하고 싶다면 most_common(N)으로 자르기
vocab = list(specials) + [w for w, _ in counter.most_common() if w not in specials]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)
print("공유 사전 크기:", VOCAB_SIZE)


공유 사전 크기: 6252


In [24]:
# 3) ID 변환 + 패딩
def encode(tokens):
    return [word2idx.get(t, word2idx[UNK]) for t in tokens]

src_ids = [encode(s) for s in src_corpus]
tgt_ids = [encode(t) for t in tgt_corpus]

def pad(seqs, pad_id=0):
    seqs = [torch.tensor(s, dtype=torch.long) for s in seqs]
    return torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=pad_id)

enc_train = pad(src_ids, pad_id=word2idx[PAD])
dec_train = pad(tgt_ids, pad_id=word2idx[PAD])
print("enc_train:", enc_train.shape)
print("dec_train:", dec_train.shape)


enc_train: torch.Size([23190, 28])
dec_train: torch.Size([23190, 32])


## Step 6. Transformer 학습

Untitled24 노트북의 Transformer 구현을 그대로 재사용합니다. (`<pad>` ID가 0이므로 마스크 함수도 그대로 동작)

In [25]:
# ===== Multi-Head Attention =====
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        b, s, _ = x.shape
        return x.view(b, s, self.num_heads, self.depth).permute(0, 2, 1, 3)

    def combine_heads(self, x):
        b, h, s, d = x.shape
        return x.permute(0, 2, 1, 3).contiguous().view(b, s, self.d_model)

    def attn(self, Q, K, V, mask):
        d_k = K.shape[-1]
        qk = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            qk = qk.masked_fill(mask == 1, float("-1e9"))  # 마스크=1 위치를 차단
        w = F.softmax(qk, dim=-1)
        return torch.matmul(w, V), w

    def forward(self, Q, K, V, mask=None):
        Qh = self.split_heads(self.W_q(Q))
        Kh = self.split_heads(self.W_k(K))
        Vh = self.split_heads(self.W_v(V))
        out, w = self.attn(Qh, Kh, Vh, mask)
        return self.linear(self.combine_heads(out)), w


class PoswiseFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


In [34]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFFN(d_model, d_ff)
        self.n1 = nn.LayerNorm(d_model, eps=1e-6)
        self.n2 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        r = x; x = self.n1(x); x, _ = self.attn(x, x, x, mask); x = self.drop(x) + r
        r = x; x = self.n2(x); x = self.ffn(x);                  x = self.drop(x) + r
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFFN(d_model, d_ff)
        self.n1 = nn.LayerNorm(d_model, eps=1e-6)
        self.n2 = nn.LayerNorm(d_model, eps=1e-6)
        self.n3 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, enc_out, self_mask, cross_mask):
        r = x; x = self.n1(x); x, _ = self.self_attn(x, x, x, self_mask);            x = self.drop(x) + r
        r = x; x = self.n2(x); x, _ = self.cross_attn(x, enc_out, enc_out, cross_mask); x = self.drop(x) + r
        r = x; x = self.n3(x); x = self.ffn(x);                                       x = self.drop(x) + r
        return x


def positional_encoding(pos, d_model):
    angle = np.array([[p / np.power(10000, 2 * (i // 2) / d_model) for i in range(d_model)] for p in range(pos)])
    angle[:, 0::2] = np.sin(angle[:, 0::2])
    angle[:, 1::2] = np.cos(angle[:, 1::2])
    return torch.FloatTensor(angle)






 # Encoder/Decoder에 final LayerNorm 추가 (Pre-LN 표준 구조)
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)  # ★ Pre-LN 표준

    def forward(self, x, mask):
        out = x; enc_attns = []
        for layer in self.enc_layers:
            out, enc_attn = layer(out, mask)
            enc_attns.append(enc_attn)
        out = self.final_norm(out)  # ★ stack 끝나고 정규화
        return out, enc_attns


class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)  # ★ Pre-LN 표준

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x; dec_attns = []; dec_enc_attns = []
        for layer in self.dec_layers:
            out, dec_attn, dec_enc_attn = layer(out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        out = self.final_norm(out)  # ★ stack 끝나고 정규화
        return out, dec_attns, dec_enc_attns


print("Encoder/Decoder 수정 완료 — final_norm 추가됨")

class Transformer(nn.Module):
    def __init__(self, vocab_size, n_layers=2, d_model=256, n_heads=8, d_ff=1024,
                 pos_len=128, dropout=0.2, shared_emb=True):
        super().__init__()
        self.d_model = d_model
        self.shared_emb = shared_emb
        self.enc_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.dec_emb = nn.Embedding(vocab_size, d_model, padding_idx=0) if not shared_emb else self.enc_emb
        self.register_buffer("pe", positional_encoding(pos_len, d_model))
        self.drop = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, vocab_size)
        self.fc.weight = self.dec_emb.weight  # weight tying (소스/타깃이 같은 사전이므로 더욱 효과적)

    def embed(self, emb, x):
        out = emb(x) * math.sqrt(self.d_model)
        out = out + self.pe[:x.size(1)].unsqueeze(0).to(x.device)
        return self.drop(out)

    def forward(self, src, tgt, enc_mask, dec_self_mask, dec_cross_mask):
        e = self.embed(self.enc_emb, src)
        for layer in self.enc_layers:
            e = layer(e, enc_mask)
        d = self.embed(self.dec_emb, tgt)
        for layer in self.dec_layers:
            d = layer(d, e, dec_self_mask, dec_cross_mask)
        return self.fc(d)


Encoder/Decoder 수정 완료 — final_norm 추가됨


In [35]:
# ===== 마스크 =====
def padding_mask(seq, pad_id=0):
    # 패딩 위치를 1로 (위 attn에서 1을 차단)
    return (seq == pad_id).float()[:, None, None, :]

def causal_mask(size, device):
    # 미래 위치를 1로
    m = torch.triu(torch.ones(size, size, device=device), diagonal=1)
    return m  # (s, s)

def make_masks(src, tgt, pad_id=0):
    enc_mask = padding_mask(src, pad_id)                                       # (B,1,1,Ls)
    cross_mask = padding_mask(src, pad_id)                                     # (B,1,1,Ls)
    dec_pad = padding_mask(tgt, pad_id)                                        # (B,1,1,Lt)
    c = causal_mask(tgt.size(1), tgt.device)[None, None, :, :]                 # (1,1,Lt,Lt)
    dec_self_mask = torch.maximum(dec_pad, c)
    return enc_mask, dec_self_mask, cross_mask


In [36]:
# ===== 학습 루프 =====
HP = dict(n_layers=2, d_model=256, n_heads=8, d_ff=1024, dropout=0.2)
WARMUP = 4000
BATCH = 64
EPOCHS = 70

pos_len = max(enc_train.size(1), dec_train.size(1)) + 2
model = Transformer(vocab_size=VOCAB_SIZE, pos_len=pos_len, **HP).to(device)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

class WarmupLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, opt, d_model, warmup):
        self.d_model, self.warmup = d_model, warmup
        super().__init__(opt)
    def get_lr(self):
        step = max(1, self.last_epoch)
        lr = (self.d_model ** -0.5) * min(step ** -0.5, step * self.warmup ** -1.5)
        return [lr for _ in self.base_lrs]

scheduler = WarmupLR(optimizer, HP["d_model"], WARMUP)
loss_fn = nn.CrossEntropyLoss(reduction="none")

def loss_with_pad_mask(real, pred):
    mask = (real != 0).float()
    pred = pred.reshape(-1, pred.size(-1))
    real_f = real.reshape(-1)
    mask_f = mask.reshape(-1)
    l = loss_fn(pred, real_f) * mask_f
    return l.sum() / mask_f.sum()

def train_step(src, tgt):
    src, tgt = src.to(device), tgt.to(device)
    gold = tgt[:, 1:]
    enc_m, dec_self_m, cross_m = make_masks(src, tgt, pad_id=0)
    optimizer.zero_grad()
    logits = model(src, tgt, enc_m, dec_self_m, cross_m)
    loss = loss_with_pad_mask(gold, logits[:, :-1])
    loss.backward()
    optimizer.step()
    scheduler.step()
    return loss.item()


파라미터 수: 5293164


In [37]:
# 학습
n = enc_train.size(0)
for ep in range(EPOCHS):
    model.train()
    idxs = list(range(0, n, BATCH))
    random.shuffle(idxs)
    total = 0.0
    pbar = tqdm(idxs, desc=f"Epoch {ep+1}/{EPOCHS}")
    for i, st in enumerate(pbar):
        s = enc_train[st:st+BATCH]
        t = dec_train[st:st+BATCH]
        loss = train_step(s, t)
        total += loss
        pbar.set_postfix(loss=f"{total/(i+1):.4f}")


Epoch 1/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 2/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 3/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 4/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 5/70:   0%|          | 0/363 [00:00<?, ?it/s]

...

Epoch 66/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 67/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 68/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 69/70:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 70/70:   0%|          | 0/363 [00:00<?, ?it/s]

In [38]:
# 추론(greedy)
@torch.no_grad()
def chat(question, max_len=30):
    model.eval()
    s = preprocess_sentence(question)
    toks = mecab.morphs(s)
    ids = [word2idx.get(t, word2idx[UNK]) for t in toks]
    src = torch.tensor([ids], dtype=torch.long, device=device)
    out = torch.tensor([[word2idx[START]]], dtype=torch.long, device=device)
    for _ in range(max_len):
        enc_m, dec_self_m, cross_m = make_masks(src, out, pad_id=0)
        logits = model(src, out, enc_m, dec_self_m, cross_m)
        nxt = logits[0, -1].argmax().item()
        out = torch.cat([out, torch.tensor([[nxt]], device=device)], dim=1)
        if nxt == word2idx[END]:
            break
    pred = [idx2word[i] for i in out[0].tolist()[1:]]
    if pred and pred[-1] == END:
        pred = pred[:-1]
    return " ".join(pred)

EXAMPLES = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야.",
]
print("=== Transformer 챗봇 답변 ===")
for q in EXAMPLES:
    print(f"Q: {q}")
    print(f"A: {chat(q)}\n")


=== Transformer 챗봇 답변 ===
Q: 지루하다, 놀러가고 싶어.
A: 좋 아 하 는 게 좋 겠 죠 .

Q: 오늘 일찍 일어났더니 피곤하다.
A: 자신 이 에요 .

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 많이 떨리 죠 .

Q: 집에 있는다는 소리야.
A: <start> <start> <start> <start> <start> <start> 좋 아 하 는 게 좋 겠 죠 .



## Step 7. 성능 측정 + RAG 스타일 retrieval 챗봇

### RAG의 적용

- 챗봇 코퍼스를 **지식 베이스(KB)** 로 사용
- 사용자 질문을 임베딩 → 코퍼스 내 가장 유사한 질문 검색 → 그 페어의 답변을 사용
- 임베딩은 추가 모델 없이 **방금 학습한 Transformer의 인코더**를 그대로 활용
  (인코더 출력의 mean pooling을 문장 임베딩으로 사용)
- 두 챗봇(생성형 Transformer vs RAG)의 BLEU를 비교

### BLEU
정답이 `ans_corpus`(원본)에 있으므로, validation으로 일부를 떼서 BLEU를 계산합니다.

In [39]:
# 1) 검증셋 분리 (원본 페어 기준)
random.seed(0)
N = len(que_corpus)
indices = list(range(N))
random.shuffle(indices)
val_size = min(500, N // 10)
val_idx = set(indices[:val_size])

val_q = [que_corpus[i] for i in sorted(val_idx)]   # 토큰 리스트
val_a = [ans_corpus[i] for i in sorted(val_idx)]   # 토큰 리스트 (원본, <start>/<end> 없음)
print("validation size:", len(val_q))


validation size: 500


In [40]:
# 2) RAG 인덱스 구축: 학습된 인코더로 모든 (원본) 질문 임베딩
@torch.no_grad()
def encode_sentence(tokens):
    """문장 토큰 리스트 → (d_model,) 임베딩 (인코더 출력의 평균)"""
    model.eval()
    ids = [word2idx.get(t, word2idx[UNK]) for t in tokens] or [word2idx[UNK]]
    src = torch.tensor([ids], dtype=torch.long, device=device)
    enc_m = padding_mask(src, pad_id=0)
    e = model.embed(model.enc_emb, src)
    for layer in model.enc_layers:
        e = layer(e, enc_m)
    # 패딩이 없으니 그냥 평균
    vec = e.mean(dim=1).squeeze(0)
    return vec / (vec.norm() + 1e-9)

# 학습에 쓰인 원본(que_corpus) 중 validation에 포함되지 않은 것만 KB로
kb_q_tokens, kb_a_tokens = [], []
for i, (q, a) in enumerate(zip(que_corpus, ans_corpus)):
    if i in val_idx:
        continue
    kb_q_tokens.append(q)
    kb_a_tokens.append(a)

print("KB size:", len(kb_q_tokens))
kb_vecs = torch.stack([encode_sentence(q) for q in tqdm(kb_q_tokens, desc="indexing")])
print("kb_vecs:", kb_vecs.shape)


KB size: 7230


indexing:   0%|          | 0/7230 [00:00<?, ?it/s]

kb_vecs: torch.Size([7230, 256])


In [41]:
# 3) RAG 챗봇: 가장 유사한 질문의 답변을 반환
@torch.no_grad()
def chat_rag(question_text, top_k=1):
    q_tokens = mecab.morphs(preprocess_sentence(question_text))
    qv = encode_sentence(q_tokens)
    sims = kb_vecs @ qv             # cosine (이미 정규화됨)
    top = torch.topk(sims, k=top_k)
    if top_k == 1:
        i = top.indices.item()
        return " ".join(kb_a_tokens[i])
    return [(" ".join(kb_a_tokens[i.item()]), float(s)) for i, s in zip(top.indices, top.values)]


print("=== RAG 챗봇 답변 ===")
for q in EXAMPLES:
    print(f"Q: {q}")
    print(f"A: {chat_rag(q)}\n")


=== RAG 챗봇 답변 ===
Q: 지루하다, 놀러가고 싶어.
A: 용기 를 내 서 말 해 보 세요 .

Q: 오늘 일찍 일어났더니 피곤하다.
A: 내려놓 는 게 가장 좋 은 방법 이 죠 .

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 마음 이 복잡 할 거 같 아요 .

Q: 집에 있는다는 소리야.
A: 비용 을 따져 보 세요 .



In [42]:
# 4) BLEU 계산
import nltk
nltk.download("punkt", quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
smooth = SmoothingFunction().method1

def calculate_bleu(predictions, references):
    """
    predictions: 토큰 리스트의 리스트 (모델 출력)
    references:  토큰 리스트의 리스트 (정답)  ← 각 샘플당 하나의 reference
    BLEU-1 ~ BLEU-4 corpus BLEU 반환
    """
    refs = [[r] for r in references]
    scores = {}
    for n in range(1, 5):
        w = tuple([1.0 / n] * n + [0.0] * (4 - n))
        scores[f"BLEU-{n}"] = corpus_bleu(refs, predictions, weights=w, smoothing_function=smooth)
    return scores


In [43]:
# 5) 두 챗봇에 대해 validation BLEU 측정
def predict_transformer(q_tokens):
    txt = " ".join(q_tokens)             # 이미 토큰화된 상태이므로 join해 입력
    out = chat(txt)
    return out.split()

def predict_rag(q_tokens):
    txt = " ".join(q_tokens)
    return chat_rag(txt).split()

preds_tf = [predict_transformer(q) for q in tqdm(val_q, desc="Transformer 추론")]
preds_rag = [predict_rag(q)         for q in tqdm(val_q, desc="RAG 추론")]

bleu_tf  = calculate_bleu(preds_tf,  val_a)
bleu_rag = calculate_bleu(preds_rag, val_a)

print("=== BLEU (validation) ===")
print("Transformer:", {k: f"{v:.4f}" for k, v in bleu_tf.items()})
print("RAG        :", {k: f"{v:.4f}" for k, v in bleu_rag.items()})


Transformer 추론:   0%|          | 0/500 [00:00<?, ?it/s]

RAG 추론:   0%|          | 0/500 [00:00<?, ?it/s]

=== BLEU (validation) ===
Transformer: {'BLEU-1': '0.1841', 'BLEU-2': '0.0837', 'BLEU-3': '0.0494', 'BLEU-4': '0.0308'}
RAG        : {'BLEU-1': '0.1998', 'BLEU-2': '0.0955', 'BLEU-3': '0.0589', 'BLEU-4': '0.0377'}


In [44]:
# 6) 예문에 대해 두 챗봇 답변 나란히 비교
print("=== 예문 비교 ===")
for q in EXAMPLES:
    print(f"Q   : {q}")
    print(f"TF  : {chat(q)}")
    print(f"RAG : {chat_rag(q)}")
    print()


=== 예문 비교 ===
Q   : 지루하다, 놀러가고 싶어.
TF  : 좋 아 하 는 게 좋 겠 죠 .
RAG : 용기 를 내 서 말 해 보 세요 .

Q   : 오늘 일찍 일어났더니 피곤하다.
TF  : 자신 이 에요 .
RAG : 내려놓 는 게 가장 좋 은 방법 이 죠 .

Q   : 간만에 여자친구랑 데이트 하기로 했어.
TF  : 많이 떨리 죠 .
RAG : 마음 이 복잡 할 거 같 아요 .

Q   : 집에 있는다는 소리야.
TF  : <start> <start> <start> <start> <start> <start> 좋 아 하 는 게 좋 겠 죠 .
RAG : 비용 을 따져 보 세요 .



## 정리

- **Transformer 챗봇**: 생성형. 데이터 외 표현을 만들 수 있지만 데이터가 1만 개 수준이라 짧은 모델/짧은 학습으로도 과적합/허튼 답변에 취약.
- **RAG 챗봇**: 학습된 인코더를 임베딩으로 활용한 retrieval. 코퍼스에 비슷한 질문이 있으면 매우 자연스러운 답을 반환하지만, 코퍼스를 벗어난 질문에는 약함.
- BLEU만 보면 데이터가 작고 답변이 다양할 때 보통 **RAG가 더 높게 나오는 경향**이 있습니다 (validation 답이 KB 답변과 정확히 같을 수 있어서). 그러나 일반화 측면에서는 다른 해석이 필요합니다.

하이퍼파라미터는 위 셀의 `HP`, `EPOCHS`, `BATCH`, `WARMUP`을 수정해 자유롭게 튜닝하세요.